# Open-Source Generative Image Pipeline
## Geração local de imagens com Stable Diffusion, Diffusers e LoRA

### Projeto pessoal de Engenharia de IA

Este projeto explora a construção de um pipeline de **IA Generativa para geração de imagens em ambiente local**, utilizando modelos open-weights executados em GPU.

O case simula uma aplicação de e-commerce para geração de criativos publicitários de acessórios para cães, com foco em:

- geração **text-to-image** com Stable Diffusion;
- execução local com **PyTorch + CUDA**;
- uso da biblioteca **Hugging Face Diffusers**;
- controle de reprodutibilidade através de **random seeds**;
- avaliação de consistência visual entre múltiplas gerações;
- aplicação de **LoRA (Low-Rank Adaptation)** para personalização de modelos;
- análise de trade-offs entre soluções open-source e APIs proprietárias;
- avaliação de requisitos de **GPU, custo, privacidade, licença e uso comercial**.

O objetivo não é apenas gerar imagens, mas demonstrar decisões de **AI Engineering** relacionadas a inferência, infraestrutura, adaptação de modelos e governança.


## Visão geral da arquitetura

```text
Prompt / Briefing
       │
       ▼
Hugging Face Diffusers
       │
       ▼
Stable Diffusion / SD-Turbo
       │
       ▼
PyTorch + CUDA
       │
       ▼
Inferência em GPU
       │
       ├── Seed control
       ├── Prompt variations
       └── LoRA adaptation
       │
       ▼
Generated Images
       │
       ▼
Avaliação de qualidade, consistência,
licença, custo e aplicabilidade
```

### Tecnologias utilizadas

| Tecnologia | Aplicação no projeto |
|---|---|
| **Python** | Orquestração do pipeline e lógica de geração |
| **PyTorch** | Execução dos modelos e gerenciamento de GPU |
| **CUDA** | Aceleração da inferência em GPU |
| **Hugging Face Diffusers** | Carregamento e execução dos pipelines de difusão |
| **Transformers** | Dependência do ecossistema Hugging Face para componentes dos modelos |
| **Accelerate** | Suporte à execução otimizada dos modelos |
| **Stable Diffusion / SD-Turbo** | Modelo open-weights para geração text-to-image |
| **LoRA** | Adaptação eficiente do modelo para consistência de um sujeito específico |
| **Pillow (PIL)** | Manipulação e composição das imagens geradas |
| **Google Colab** | Ambiente de execução com GPU |
| **Hugging Face Hub** | Distribuição e carregamento dos pesos dos modelos |

### Competências demonstradas

**Generative AI • Diffusion Models • Stable Diffusion • LoRA • PyTorch • CUDA • Hugging Face • Prompt Engineering • Model Inference • Open-Weights Models • AI Engineering • GPU Computing**


## Requisitos de execução, download e licença

> **Este notebook precisa de GPU.** No Colab: **Ambiente de execução → Alterar o tipo de ambiente → GPU T4** (disponível no plano gratuito). A primeira célula confere.

> **Download de ~2,5 GB na primeira execução.** Os pesos do modelo vêm do Hugging Face e ficam em cache na sessão. No Colab o disco é efêmero: fechou a sessão, o download some (não fica no seu computador).

> **Licença: SD-Turbo é NÃO comercial** (licença de pesquisa da Stability). Para *vender* o criativo, troque por `stable-diffusion-xl-base-1.0` ou `FLUX.1-schnell` (Apache 2.0). Esse ponto é essencial para uso responsável em produção.

> **Custo:** aqui você não paga por imagem — paga por GPU. No Colab Free, a GPU é "de graça" (com filas e limites). Em produção, GPU é a linha mais cara da conta.

> **Princípio de engenharia:** avaliar qualidade, custo, infraestrutura, licença e reprodutibilidade antes de escolher um modelo.


## Objetivos técnicos

Este projeto foi estruturado para responder a cinco perguntas de engenharia:

1. Como executar um modelo generativo de imagem sem depender de uma API externa?
2. Quais requisitos de infraestrutura são necessários para inferência local?
3. Como tornar os experimentos reproduzíveis?
4. Como melhorar a consistência de um sujeito ou produto utilizando LoRA?
5. Quais trade-offs existem entre custo, controle, qualidade, privacidade e licença?

A implementação utiliza inferência em **FP16** para reduzir consumo de VRAM e permitir execução em GPUs com recursos limitados.


In [ ]:
# Versoes testadas em 07/2026; se algo falhar no futuro, remova os pinos (==).
%pip install -q "diffusers==0.35.1" "transformers==4.56.1" "accelerate==1.10.1"
%pip uninstall -y -q torchao   # a versao do Colab conflita com o peft, usado pelo LoRA na 3.6
# No Colab o torch ja vem instalado (com CUDA). Fora do Colab, instale o torch certo em pytorch.org.
# O ERROR de "dependency resolver" que aparece abaixo e esperado no Colab e nao afeta a aula.

## 1. Validação do ambiente de GPU

**Por que:** difusão local roda em GPU. Se a célula abaixo disser que não há CUDA, ajuste o ambiente antes de continuar — nada do resto funciona sem isso.


In [ ]:
import torch  # PyTorch: e ele quem enxerga a GPU

print("CUDA disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("SEM GPU. No Colab: Ambiente de execucao > Alterar o tipo de ambiente > GPU T4.")
    print("Depois de trocar, rode o notebook desde o inicio.")

In [ ]:
# Pre-flight: interrompe cedo com uma mensagem clara se o runtime não tiver GPU.
if not torch.cuda.is_available():
    raise RuntimeError("Ative uma GPU no Colab: Ambiente de execução > Alterar tipo de ambiente > GPU T4.")
print("Pre-flight OK: GPU CUDA pronta para geração local.")


### Carregamento dos pesos e uso de cache

Não existe uma linha de "download": o `from_pretrained("repo_id")` baixa os pesos do Hugging Face na **primeira execução** e guarda em cache. Depois, lê do cache.

| Modelo | Download | Passos | Licença |
|---|---|---|---|
| **SD-Turbo** (usado aqui) | ~2,5 GB | **1** | não comercial (pesquisa) |
| SDXL base 1.0 | ~7 GB | ~30 | **comercial OK** |
| FLUX.1-schnell | ~33 GB | 4 | **Apache 2.0 (comercial OK)** |

Escolhemos o SD-Turbo porque **cabe na T4 gratuita e gera em 1 passo** — dá para rodar ao vivo em aula. Os outros dois são os candidatos de produção.


In [ ]:
# TROCAVEL — confira licença e requisito de GPU de cada peso antes de adotar:
#   "stabilityai/sd-turbo"                      -> usado aqui (~2,5 GB, 1 passo, NAO comercial)
#   "stabilityai/stable-diffusion-xl-base-1.0"  -> comercial OK, ~7 GB, ~30 passos (mais lento)
#   "black-forest-labs/FLUX.1-schnell"          -> Apache 2.0, melhor texto, ~33 GB (nao cabe na T4 free)
MODEL_IMAGE = "stabilityai/sd-turbo"
print("Modelo escolhido:", MODEL_IMAGE)

## 2. Definição do briefing e engenharia de prompt

**Por que:** a lição escondida deste colab. Modelos abertos foram treinados com legendas majoritariamente em **inglês**; o mesmo briefing em PT rende visivelmente menos. As APIs fechadas mascaram isso de você — o modelo aberto, não.


In [ ]:
BRIEFING_PT = (
    "Fotografia publicitária premium para e-commerce de um acessório para Golden Retriever. "
    "Um Golden Retriever adulto, saudável e amigável usa um peitoral ergonômico premium "
    "azul-petróleo, com tiras ajustáveis e acabamento refletivo. Fundo de estúdio bege claro, "
    "iluminação suave e sofisticada, produto em destaque, espaço livre no topo para texto da campanha. "
    "Estilo limpo, moderno e realista."
)
BRIEFING_EN = (
    "Premium e-commerce advertising photo for a Golden Retriever accessory. "
    "A healthy, friendly adult Golden Retriever wears a premium teal ergonomic dog harness "
    "with adjustable straps and subtle reflective trim. Light beige studio background, "
    "soft sophisticated studio lighting, the harness clearly visible as the hero product, "
    "free space at the top for campaign copy. Clean, modern, photorealistic style."
)
print(BRIEFING_EN)


Carregue o pipeline. Na primeira vez, os ~2,5 GB descem aqui (1 a 3 minutos no Colab).


In [ ]:
from diffusers import AutoPipelineForText2Image  # pipeline de texto -> imagem

pipe = AutoPipelineForText2Image.from_pretrained(  # baixa os pesos na 1a execução (~2,5 GB)
    MODEL_IMAGE,
    torch_dtype=torch.float16,  # meia precisão: cabe na memória da T4
    safety_checker=None, requires_safety_checker=False,  # evita falso positivo em foto de produto
).to("cuda")  # manda o modelo para a GPU
print("Pipeline pronto na GPU.")

## 3. Inferência otimizada com SD-Turbo

**Por que:** o SD-Turbo é uma versão *destilada* do Stable Diffusion: aprendeu a fazer em 1 passo o caminho de volta do ruído que o modelo original faz em ~30. A receita do turbo: `num_inference_steps=1` e `guidance_scale=0.0`.


In [ ]:
imagem = pipe(prompt=BRIEFING_EN, num_inference_steps=1, guidance_scale=0.0).images[0]  # 1 passo
imagem.save("opensource_base.png")
print("salvo: opensource_base.png")
imagem  # exibe no notebook

Repare no **texto**: o modelo leve quase sempre erra o nome da marca (letras trocadas, garranchos). É o calcanhar de aquiles do aberto leve — e exatamente o eixo nº 1 do o case. Guarde essa imagem para a comparação.


## 4. Geração programática de múltiplas variações

**Por que:** aqui a conta muda de natureza. Cada imagem nova não custa nada além de segundos de GPU — em volume alto, é isso que vira argumento a favor do aberto.


In [ ]:
from IPython.display import display    # exibir várias imagens no laço

variacoes_en = [
    # Variação 1 — ângulo frontal, estúdio claro, clima premium
    "Adult Golden Retriever wearing the same premium teal ergonomic dog harness with adjustable "
    "straps and subtle reflective trim, front three-quarter view, light beige seamless studio "
    "background, soft diffused studio light, premium calm mood, clean e-commerce advertising photo, "
    "harness clearly visible, negative space at the top.",

    # Variação 2 — ângulo lateral, ambiente externo, clima ativo
    "Adult Golden Retriever wearing the same premium teal ergonomic dog harness with adjustable "
    "straps and subtle reflective trim, full side view while standing in a modern urban park, "
    "soft natural morning light, active and friendly mood, shallow depth of field, premium pet "
    "accessory campaign, harness clearly visible, negative space on the left.",

    # Variação 3 — close, fundo escuro, clima sofisticado
    "Adult Golden Retriever wearing the same premium teal ergonomic dog harness with adjustable "
    "straps and subtle reflective trim, close-up chest and head portrait from a slightly low angle, "
    "dark charcoal studio background, cinematic rim lighting, sophisticated luxury mood, photorealistic "
    "pet advertising, harness details in sharp focus, negative space on the right.",
]

for i, p in enumerate(variacoes_en, 1):
    img = pipe(prompt=p, num_inference_steps=1, guidance_scale=0.0).images[0]
    img.save(f"opensource_variacao_{i}.png")
    display(img)
    print(f"opensource_variacao_{i}.png <- {p}\n")


## 5. Reprodutibilidade com random seed

**Por que:** difusão parte de ruído aleatório, então o mesmo prompt gera uma imagem diferente a cada rodada. A **seed** fixa esse ruído inicial e torna o experimento reproduzível — requisito de qualquer teste A/B sério.

Guarde a lição para a próxima seção: seed igual com prompt igual dá a mesma imagem; seed igual com **prompt diferente** dá outra imagem. Seed é reprodutibilidade, **não** é consistência de personagem.


In [ ]:
# Duas gerações com a MESMA seed e o MESMO prompt: imagens idênticas
for _ in range(2):
    g = torch.Generator("cuda").manual_seed(42)      # troque 42 por 43 e as duas mudam juntas
    display(pipe(prompt=variacoes_en[0], num_inference_steps=1,
                 guidance_scale=0.0, generator=g).images[0])

### Decisão de engenharia: por que LoRA?

**LoRA (Low-Rank Adaptation)** permite adaptar partes do modelo sem realizar fine-tuning completo de todos os parâmetros.

Isso reduz:

- custo computacional;
- quantidade de VRAM necessária;
- tempo de treinamento;
- tamanho dos artefatos gerados.

Em um cenário real, essa abordagem pode ser utilizada para especializar um modelo em um produto, identidade visual, personagem ou domínio específico.


## 6. Personalização eficiente com LoRA

**Por que:** consistência. Em abordagens anteriores baseadas apenas em prompt o produto mudava a cada geração, e a seção anterior mostrou que seed fixa não resolve isso — ela só repete a mesma imagem. **LoRA** é um ajuste leve do modelo: treina-se com 10–30 fotos do seu produto, personagem ou estilo e o modelo passa a reproduzi-lo em qualquer cenário. Isso só existe no mundo open-weights — API fechada não se deixa ajustar.

Aqui usamos um LoRA já treinado em 5 fotos de **um** cachorro específico. O gatilho aprendido no treino é `sks dog`: um token sem sentido para o modelo base, que o LoRA transforma em "aquele cachorro". Cinco passos curtos: liberar a GPU, carregar a base SD 1.5, aplicar o LoRA, gerar três cenários com a mesma seed e montar o painel.

Dois avisos de método: treinar um LoRA exige GPU e engenharia (não é uma célula de Colab), e **a licença do LoRA conta tanto quanto a do modelo base** — cheque as duas antes de uso em produção.


In [ ]:
# Passo 1: libera a GPU — o SD-Turbo sai de cena
import gc, torch

if "pipe" in globals(): del pipe  # o turbo nao serve aqui: o LoRA e de SD 1.5
gc.collect(); torch.cuda.empty_cache()
print("GPU liberada.")

In [ ]:
# Passo 2: carrega a base que o LoRA espera — SD 1.5
from diffusers import StableDiffusionPipeline

# Estes repositórios são públicos. Se o Hugging Face pedir autenticação no futuro,
# crie gratuitamente um token em huggingface.co e faça login, mas a aula não exige API paga.
pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to("cuda")
print("Base SD 1.5 na GPU.")


In [ ]:
# Passo 3: o LoRA por cima da base. Repo de 2023 -> loader legado + arquivo .bin.
pipe.unet.load_attn_procs(
    "patrickvonplaten/lora_dreambooth_dog_example",
    weight_name="pytorch_lora_weights.bin",
)

# Sem esta linha, um LoRA ignorado passa em silêncio e você gera 3 imagens à toa:
assert any("lora" in n for n, _ in pipe.unet.named_modules()), "LoRA não aplicado"
print("LoRA aplicado. Gatilho aprendido no treino: sks dog")

In [ ]:
# Passo 4: mesmo personagem ("sks dog"), três cenários, a mesma seed em todos
SEED = 42

cenarios = [
    "a photo of sks dog wearing a teal dog harness in a light beige premium studio, product advertising",
    "a photo of sks dog wearing a teal dog harness in a modern urban park at sunrise, active lifestyle advertising",
    "a photo of sks dog wearing a teal dog harness in a dark charcoal studio with cinematic rim light, luxury advertising",
]

imagens = []
for prompt in cenarios:
    g = torch.Generator("cuda").manual_seed(SEED)   # recriar a CADA volta do laço
    imagens.append(pipe(prompt, num_inference_steps=30, generator=g).images[0])

print(len(imagens), "imagens geradas")


In [ ]:
# Passo 5: painel lado a lado — a prova visual da consistência
from PIL import Image

w, h = imagens[0].size
painel = Image.new("RGB", (w * len(imagens), h))
for i, img in enumerate(imagens):
    painel.paste(img, (i * w, 0))

painel.save("painel_lora.png")
painel

## 7. Trade-offs de uma arquitetura local

- **Privacidade / on-prem.** Briefing, produto e imagem não saem da sua infraestrutura. Em setor regulado, isso decide sozinho.
- **Custo fixo em volume alto.** API cobra por imagem; GPU custa por hora. Milhares de imagens/mês invertem a conta a favor do local.
- **Controle.** LoRA, ControlNet, inpainting fino, seed travada: o aberto obedece de um jeito que API fechada não expõe.
- **Quando NÃO compensa:** volume baixo (30/semana!), texto de marca crítico, ou nenhum time para cuidar de GPU, driver e fila. "Grátis" é falácia: você paga em infra e engenharia.


## 8. Comparação arquitetural: open-source vs. APIs

Comparação conceitual entre três abordagens de geração:

| Eixo | gpt-image (Colab 1) | Gemini image (Colab 2) | SD-Turbo local (este) |
|---|---|---|---|
| **Texto de marca na imagem** | Muito forte | Muito forte | Fraco (FLUX melhora) |
| **Custo** | Centavos/imagem | Centavos/imagem + free tier | Zero por imagem; paga GPU |
| **Controle (LoRA etc.)** | Prompt + referência | Prompt + edição multi-turno | **LoRA/ControlNet: trava a marca** |
| **Licença/uso comercial** | Outputs seus (sob política) | OK, com SynthID sempre | **Depende do peso** (turbo = não comercial) |
| **Latência/setup** | Zero setup, segundos | Zero setup, segundos | GPU, driver, download de GB |
| **Privacidade** | Cloud-only | Cloud-only | **100% local/on-prem** |

> Não existe "melhor gerador" no absoluto — existe o melhor **para a restrição do case**. Aqui, texto de marca legível derruba o aberto leve; privacidade e volume alto derrubam as APIs. Híbrido (base no fechado + refino local) é comum.


## Critérios de avaliação do projeto

Além da qualidade visual, uma solução de geração de imagens em produção deve ser avaliada por critérios objetivos:

| Critério | O que avaliar |
|---|---|
| **Latência** | Tempo médio de inferência por imagem |
| **VRAM** | Memória necessária para carregar e executar o modelo |
| **Throughput** | Quantidade de imagens geradas por unidade de tempo |
| **Reprodutibilidade** | Capacidade de reproduzir resultados com seed controlada |
| **Consistência** | Preservação de produto, sujeito ou estilo entre gerações |
| **Qualidade visual** | Artefatos, coerência, realismo e aderência ao prompt |
| **Custo** | GPU/hora versus cobrança por imagem de APIs externas |
| **Privacidade** | Permanência dos dados dentro da infraestrutura |
| **Licença** | Permissão para pesquisa, redistribuição e uso comercial |

Em uma evolução futura, esses indicadores podem ser persistidos em um experimento estruturado para comparação entre modelos.


## 9. Limitações e riscos técnicos

- **Texto na imagem.** O aberto leve erra o nome da marca — no o case, isso desclassifica o SD-Turbo sozinho.
- **Licença.** SD-Turbo é não comercial; usar em campanha real é violação. Cheque modelo E LoRA antes de uso em produção.
- **"Grátis" que custa caro.** GPU, drivers, VRAM, filas, atualização de pesos: o custo migra de API para infra + engenharia.
- **Qualidade datada no leve.** SD-Turbo é rápido, não é fronteira. Para competir em qualidade, o aberto pede FLUX/SDXL — e aí a GPU gratuita já não basta.


Opcional: liberar a memória da GPU (útil se for carregar outro modelo na mesma sessão). No Colab, o disco e a memória somem sozinhos quando a sessão fecha.


In [ ]:
# Opcional: libera a memória da GPU nesta sessão.
import gc
for v in ["pipe"]:
    if v in globals():
        del globals()[v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Memoria da GPU liberada.")

## 10. Extensão do projeto: geração de criativos sem API externa



### Cenário



Uma organização com requisitos elevados de privacidade precisa gerar criativos sem enviar prompts ou imagens para APIs externas.



A solução abaixo demonstra como o mesmo pipeline pode ser executado integralmente em infraestrutura própria.



### Implementação



- briefing em inglês para melhorar aderência ao modelo;

- geração de múltiplas variações;

- avaliação da consistência do produto;

- análise de legibilidade de texto;

- validação da licença antes de qualquer uso comercial;

- avaliação de custo e privacidade.



Esse cenário aproxima o notebook de uma decisão real de arquitetura de IA, em vez de tratar geração de imagens apenas como experimentação visual.



In [ ]:
# TODO 1: briefing em inglês — setor de acessórios para cães
meu_briefing_en = (
    "Premium e-commerce advertising photo for a Golden Retriever accessory. "
    "A healthy adult Golden Retriever wears a premium teal ergonomic dog harness with adjustable "
    "straps and subtle reflective trim. The harness is the hero product and must remain visually "
    "consistent across all variations. Photorealistic fur and product materials, professional "
    "commercial lighting, clean composition, and negative space reserved for campaign copy."
)

# TODO 2: três variações locais — muda ângulo, fundo e clima, mantendo o mesmo conceito/produto
meus_prompts_en = [
    meu_briefing_en + (
        " Front three-quarter view, light beige seamless studio background, soft diffused light, "
        "calm premium mood, negative space at the top."
    ),
    meu_briefing_en + (
        " Full side view, modern urban park background with gentle bokeh, natural morning light, "
        "active friendly mood, negative space on the left."
    ),
    meu_briefing_en + (
        " Close-up chest and head portrait from a slightly low angle, dark charcoal studio background, "
        "cinematic rim light, sophisticated luxury mood, negative space on the right."
    ),
]

# Se a seção de LoRA foi executada antes, ela substitui 'pipe' por SD 1.5.
# Para este exercício de 1 passo, recarregamos o SD-Turbo explicitamente.
import gc, torch
from diffusers import AutoPipelineForText2Image
from IPython.display import display

if "pipe" in globals():
    del pipe
gc.collect()
torch.cuda.empty_cache()

pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_IMAGE,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
).to("cuda")

for i, p in enumerate(meus_prompts_en, 1):
    img = pipe(prompt=p, num_inference_steps=1, guidance_scale=0.0).images[0]
    nome = f"exercicio_local_{i}.png"
    img.save(nome)
    display(img)
    print(f"{nome} <- {p}\n")

print("Briefing e 3 variações concluídos.")


### Análise do modelo e risco

**Legibilidade do texto:** o SD-Turbo não é confiável para gerar logotipos ou textos de marca perfeitamente legíveis. Por isso, o briefing reserva espaço negativo para o texto da campanha, que deve ser aplicado depois em uma ferramenta de design.

**Modelo para venda:** para uma campanha comercial, eu não usaria o SD-Turbo deste exercício sem antes validar cuidadosamente sua licença de uso. Para produção, escolheria um modelo open-weights cuja licença permita uso comercial e documentaria a versão e a licença adotadas antes da publicação.

**Risco concreto:** a aparência do peitoral pode mudar entre as variações — cor, formato das tiras, fechos e detalhes refletivos podem ser alterados ou inventados pelo modelo. Isso pode gerar uma peça publicitária visualmente atraente, mas que não representa fielmente o produto real.

**Mitigação:** usar imagens reais do produto como referência e uma etapa obrigatória de curadoria humana; padronizar o prompt e a seed durante testes; quando necessário, usar técnicas de controle como LoRA/ControlNet/IP-Adapter ou inpainting para aumentar a consistência. Logotipo, preço e textos devem ser inseridos posteriormente em software de design, em vez de depender do texto gerado pelo modelo.

**Conclusão:** neste case de cerca de 30 criativos por semana e exigência on-premises, rodar localmente pode compensar principalmente pela privacidade e pelo custo marginal baixo por imagem. A contrapartida é assumir custos de GPU, manutenção e curadoria, então a decisão depende da infraestrutura e do nível de consistência exigido.


## Conclusões do projeto

Este projeto demonstra uma implementação completa de inferência generativa com modelos open-weights e evidencia aspectos importantes de **AI Engineering**:

- configuração e validação de ambiente acelerado por GPU;
- carregamento de modelos através do Hugging Face Hub;
- inferência otimizada em meia precisão;
- controle de aleatoriedade e reprodutibilidade;
- geração programática de múltiplos outputs;
- personalização eficiente com LoRA;
- análise de custo, privacidade e licença;
- identificação de limitações de qualidade e consistência.

### Evoluções futuras

Como próximos passos, o projeto pode ser expandido com:

- **ControlNet** para controle estrutural da geração;
- **inpainting** para edição localizada;
- avaliação de modelos mais recentes como **SDXL** e **FLUX**;
- registro estruturado de experimentos;
- medição automática de latência e consumo de VRAM;
- criação de uma API de inferência com **FastAPI**;
- containerização com **Docker**;
- deploy em infraestrutura GPU;
- observabilidade do serviço e monitoramento de custo por geração.

### Posicionamento de portfólio

Este notebook foi organizado como um projeto pessoal de **Engenharia de IA Generativa**, demonstrando não apenas uso de modelos de difusão, mas também decisões relacionadas a infraestrutura, adaptação, reprodutibilidade, governança e operação de modelos.


In [ ]:
# Empacota apenas os arquivos realmente gerados.
from pathlib import Path
import zipfile

arquivos = sorted(Path(".").glob("*.png"))
if not arquivos:
    raise RuntimeError(
        "Nenhuma imagem PNG foi encontrada. Execute primeiro as células de geração de imagem."
    )

with zipfile.ZipFile("entrega_colab3.zip", "w", compression=zipfile.ZIP_DEFLATED) as z:
    for arquivo in arquivos:
        z.write(arquivo, arcname=arquivo.name)

print(f"Pronto: entrega_colab3.zip com {len(arquivos)} imagem(ns) — baixe pelo ícone de pasta na barra lateral.")
